# Introduction to Classification
### Hands-on Notebook — Logistic Regression & Probability-Based Classification

**Learning Objectives**
1. Predict categories using **Logistic Regression**
2. Understand the **probability-based classification logic**

**Subtopics:** Classification Models · Logistic Regression

---
**The situation:** you have each learner's practice hours and whether they passed an assessment (Pass/Fail). Unlike last session, the target isn't a number — it's a category. Let's see why a straight line fails here, and how Logistic Regression fixes it.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix

plt.rcParams['figure.figsize'] = (7, 4.5)
np.random.seed(42)


## 1. The dataset

Hours practiced vs Pass(1)/Fail(0) — the same worked example from the slides, extended slightly.


In [ ]:
demo = pd.DataFrame({
    "hours": [1, 2, 3, 4, 5, 6, 7],
    "passed": [0, 0, 0, 1, 1, 1, 1],
})
demo


## 2. Why not just fit a straight line?

Let's actually try it, to see the problem instead of just being told about it.


In [ ]:
lin_model = LinearRegression()
lin_model.fit(demo[["hours"]], demo["passed"])

hours_range = np.linspace(0, 9, 100).reshape(-1, 1)
lin_predictions = lin_model.predict(hours_range)

fig, ax = plt.subplots()
ax.scatter(demo["hours"], demo["passed"], color="#F3F7FB", edgecolor="#122238", s=70, zorder=3, label="Actual (0/1)")
ax.plot(hours_range, lin_predictions, color="#E15B64", linewidth=2, label="Linear regression fit")
ax.axhline(0, color="#5F7996", linewidth=1, linestyle="--")
ax.axhline(1, color="#5F7996", linewidth=1, linestyle="--")
ax.set_xlabel("Hours practiced")
ax.set_ylabel("Passed")
ax.set_title("A straight line predicts values outside [0, 1]")
ax.legend()
plt.tight_layout()
plt.show()

print("Predicted 'probability' at 0 hours:", round(lin_model.predict([[0]])[0], 2))
print("Predicted 'probability' at 9 hours:", round(lin_model.predict([[9]])[0], 2))


Notice the predictions go below 0 and above 1 — nonsensical for a probability. This is exactly why classification needs a different function: the **sigmoid**.


## 3. The sigmoid function — from scratch

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

No matter what real number goes in, the output is always squashed into (0, 1).


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.linspace(-8, 8, 200)
p_values = sigmoid(z_values)

fig, ax = plt.subplots()
ax.plot(z_values, p_values, color="#F2A93B", linewidth=2.5)
ax.axhline(0.5, color="#5F7996", linewidth=1, linestyle="--")
ax.set_xlabel("z (linear score)")
ax.set_ylabel("p (probability)")
ax.set_title("The sigmoid function")
plt.tight_layout()
plt.show()

print("sigmoid(-4.5) =", round(sigmoid(-4.5), 3))
print("sigmoid(0)    =", round(sigmoid(0), 3))
print("sigmoid(3)    =", round(sigmoid(3), 3))


## 4. From linear score to probability — the worked example

Using the slide's coefficients: **b₀ = −6, b₁ = 1.5**, so `z = -6 + 1.5 * hours`.


In [ ]:
b0, b1 = -6, 1.5

demo_worked = pd.DataFrame({"hours": [1, 2, 4, 5, 6]})
demo_worked["z"] = b0 + b1 * demo_worked["hours"]
demo_worked["p"] = sigmoid(demo_worked["z"])
demo_worked["prediction"] = np.where(demo_worked["p"] >= 0.5, "Pass", "Fail")
demo_worked


**Checkpoint:** this should match the slide table — 4 hours sits almost exactly on the p = 0.5 decision boundary.


## 5. Fitting a real Logistic Regression model

Now let's let scikit-learn actually *learn* the coefficients from data, instead of us specifying them.


In [ ]:
clf = LogisticRegression()
clf.fit(demo[["hours"]], demo["passed"])

print("Learned b0 (intercept):", clf.intercept_[0])
print("Learned b1 (coefficient):", clf.coef_[0][0])

# predicted probabilities for each original data point
demo["predicted_prob"] = clf.predict_proba(demo[["hours"]])[:, 1]
demo["predicted_class"] = clf.predict(demo[["hours"]])
demo


In [ ]:
hours_range = np.linspace(0, 9, 200).reshape(-1, 1)
prob_curve = clf.predict_proba(hours_range)[:, 1]

fig, ax = plt.subplots()
ax.scatter(demo["hours"], demo["passed"], color="#F3F7FB", edgecolor="#122238", s=70, zorder=3, label="Actual (0/1)")
ax.plot(hours_range, prob_curve, color="#F2A93B", linewidth=2.5, label="Predicted probability")
ax.axhline(0.5, color="#5F7996", linewidth=1, linestyle="--", label="threshold = 0.5")
ax.set_xlabel("Hours practiced")
ax.set_ylabel("Probability of passing")
ax.set_title("Logistic Regression fit")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Decision boundary & accuracy

Where does `predicted probability = 0.5`? That's the decision boundary — the point where the model flips its prediction.


In [ ]:
# Solve for hours where z = 0  ->  b0 + b1*hours = 0
boundary_hours = -clf.intercept_[0] / clf.coef_[0][0]
print(f"Decision boundary at ~{boundary_hours:.2f} hours")

preds = clf.predict(demo[["hours"]])
acc = accuracy_score(demo["passed"], preds)
print(f"Training accuracy: {acc:.2f}")

cm = confusion_matrix(demo["passed"], preds)
print("Confusion matrix:\n", cm)


## 7. Exercises

Work through these before checking the solutions.


### Exercise 1 — Compute probability by hand
Using the worked-example coefficients (b₀ = −6, b₁ = 1.5), what's the predicted probability of passing for a learner who practiced **3.5 hours**? Would the model predict Pass or Fail?


In [ ]:
# TODO: compute z, then p = sigmoid(z), for hours = 3.5



<details><summary>Show solution</summary>

```python
z = -6 + 1.5 * 3.5
p = sigmoid(z)
print(z, p)   # z = -0.75, p ≈ 0.32 -> Fail
```
</details>


### Exercise 2 — Move the threshold
Using the fitted `clf` model, what predictions do you get if the threshold is **0.7** instead of 0.5 (i.e., only predict Pass if probability ≥ 0.7)? How does that change the predicted classes for the `demo` dataset?


In [ ]:
# TODO: use demo['predicted_prob'] and a threshold of 0.7 to build new predictions



<details><summary>Show solution</summary>

```python
demo["predicted_class_0.7"] = np.where(demo["predicted_prob"] >= 0.7, 1, 0)
print(demo[["hours", "passed", "predicted_prob", "predicted_class_0.7"]])
```
Raising the threshold makes the model more conservative about predicting "Pass" — some learners who would have been predicted Pass at 0.5 now fall below 0.7 and get predicted Fail. This is the probability-based logic in action: the model's output doesn't change, only how we act on it does.
</details>


### Exercise 3 — A new learner
A new learner practiced for **3 hours**. Using the fitted `clf` model (not the hand-coded coefficients), what's their predicted probability of passing, and what class would the model assign?


In [ ]:
# TODO: use clf.predict_proba and clf.predict on [[3]]



<details><summary>Show solution</summary>

```python
prob = clf.predict_proba([[3]])[0][1]
pred = clf.predict([[3]])[0]
print(f"Probability of passing: {prob:.3f}")
print(f"Predicted class: {'Pass' if pred == 1 else 'Fail'}")
```
</details>


## 8. Recap

- A straight line is the wrong tool for 0/1 outcomes — it isn't bounded, and can predict nonsensical values.
- **Logistic Regression** fixes this by passing a linear score `z` through the **sigmoid**, producing a valid probability between 0 and 1.
- A **threshold** (0.5 by default) turns that probability into a final category — the model estimates likelihood, the threshold makes the call.
- Moving the threshold changes which errors you're more willing to make — more on this trade-off (and metrics like precision/recall) in a later session.

**Next up:** evaluating classification models — accuracy, precision, recall, and the confusion matrix in depth.
